In [2]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import db_dtypes
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [3]:
sql = f"""
WITH payment_sessions AS (
  SELECT
    DeviceId,
    SessionId,
    MIN(Time) AS first_payment_time
  FROM `openrice-production.ORGA.PV_20260915`
  WHERE DeviceId IS NOT NULL
    AND SessionId IS NOT NULL
    AND (
      LOWER(EventAction) LIKE '%takeaway.pay%'
      OR LOWER(EventLabelRaw) LIKE '%takeaway.pay%'
    )
  GROUP BY DeviceId, SessionId
  ORDER BY first_payment_time
  LIMIT 20
)

SELECT pv.*
FROM `openrice-production.ORGA.PV_20260915` AS pv
INNER JOIN payment_sessions AS payment
  ON pv.DeviceId = payment.DeviceId
  AND pv.SessionId = payment.SessionId
ORDER BY payment.first_payment_time, pv.DeviceId, pv.SessionId, pv.Time;
"""

#Execute query
df_bq = client.query(sql).result().to_dataframe()
#df_bq

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
#df_bq.to_csv("output.csv", index=False, encoding="utf-8-sig")


In [ ]:
import pandas as pd
import json

event_mapping = {
        "or.app.start": "開啟 App",
        "or.app.resume": "重新載入 App",
        "or.qcksearch": "開啟快速搜尋",
        "or.search.quick": "搜尋餐廳",
        "or.search.layer": "搜尋頁面",
        "or.search.layer.record": "使用搜尋紀錄",
        "or.search.get-poi": "從搜尋結果進入 POI",
        "impression.poi": "瀏覽POI",
        "or.poi.get-details": "載入 POI 詳情",
        "or.poi.get-overview": "載入 POI 概覽",
        "or.poi.back": "離開 POI 頁面",
        "or.takeaway.order": "進入外賣自取點餐頁/菜單",
        "poi.emenu.get-item": "選擇食物",
        "or.takeaway.checkout": "前往結帳",
        "or.takeaway.place-order": "確認下單",
        "or.takeaway.pay": "付款",
        "user.bookmark.poi": "新增收藏",
        "or.myor.bkmpoi": "點擊「已收藏」",
        "myor.search.bkmpoi": "瀏覽收藏POI",
        "or.search.themelist.takeaway": "點擊外賣自取按鈕",

        "or.myor.orderlist": "載入「我的訂單」列表",
        "or.poi.get-photos.food": "載入餐廳食物相片",
        "or.search.layer.tips": "使用搜尋提示",
        "or.poi.get-photos.menu": "載入餐廳菜單相片",
        "or.app.openpush": "從 push notification 開啟 App",
        "myor.search.bkmpoi": "從收藏清單搜尋收藏 POI ",
        "user.review.write": "撰寫評論",
        "user.myor": "點擊「我的」",
        "or.poi.get-reviews": "載入餐廳評論",
        "or.poi.review.back": "離開餐廳評論",
        "or.search.nearby": "附近餐廳搜尋",
       "impression.sponsor.poi": "瀏覽贊助餐廳",
        "or.poi.map": "查看餐廳地圖",
        "or.myor.bkmpoi": "從收藏清單選擇餐廳",
        "or.takeaway.view-basket": "查看外賣購物籃",
        "or.deeplink.get-poi": "由 deep link 獲取 POI",
        "view.sr1.promotion": "查看搜尋結果中的廣告",
        "or.poi.photo.detail": "查看餐廳相片詳情",
        "user.share.poi": "分享餐廳",

        "or.bookmark.get-poi": "點擊收藏POI",
        "or.search.layer.search": "使用搜尋頁面分類列表",  # whatkey:下午茶，米芝蓮 ect
        "user.unbookmark.poi": "取消收藏",
        "or.search.back": "離開搜尋頁面",
        "or.takeaway.view-basket": "查看外賣購物籃"

}

def classify_event(event_action):
    action = str(event_action).strip().lower()
    return event_mapping.get(action, "其他事件")

# 時間排序，避免同一 session 的 event 順序混亂
tracking_source = df_bq.copy()
tracking_source["Time"] = pd.to_datetime(tracking_source["Time"])
tracking_source = tracking_source.sort_values(["Time"])

# 保留 EventAction 原文，並產生對應的行為判斷
tracking_source["event_action_raw"] = (
    tracking_source["EventAction"]
    .fillna("(empty event action)")
    .astype(str)
)

tracking_source["event_journey"] = (
    tracking_source["EventAction"]
    .apply(classify_event)
)

# 每個 DeviceId + SessionId 一行；timeline 與 journey 的相同 index 互相對應
behavior_tracking = (
    tracking_source
    .groupby(["SessionId", "DeviceId"], as_index=False, sort=False)
    .agg(
        event_timeline=(
            "event_action_raw",
            lambda events: json.dumps(
                events.tolist(),
                ensure_ascii=False,
                indent=2
            )
        ),
        journey=(
            "event_journey",
            lambda indicators: json.dumps(
                indicators.tolist(),
                ensure_ascii=False,
                indent=2
            )
        ),
    )
)

behavior_tracking



,SessionId,DeviceId,event_timeline,journey
0,178112,2c1ec027-be02-4201-a8ce-09c4ca6031b2,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""o...","[\n ""重新載入 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ..."
1,127050,1f894cab868b116f,"[\n ""or.poi.get-overview"",\n ""or.poi.get-det...","[\n ""載入 POI 概覽"",\n ""載入 POI 詳情"",\n ""載入 POI 詳..."
2,925466,e5f59098851843a7,"[\n ""or.app.start"",\n ""or.app.start"",\n ""or...","[\n ""開啟 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ""搜..."
3,119842,1daf763b-8da1-4b2f-b824-12d558a8ffda,"[\n ""or.app.resume"",\n ""impression.poi"",\n ...","[\n ""重新載入 App"",\n ""瀏覽POI"",\n ""瀏覽POI"",\n ""瀏..."
4,252253,3ea40686-e25e-4ccb-9887-64c71db37c78,"[\n ""or.app.resume"",\n ""or.qcksearch"",\n ""o...","[\n ""重新載入 App"",\n ""開啟快速搜尋"",\n ""載入 POI 詳情"",\..."
5,818327,cb512dc0-6f1c-4824-8d2e-e08565916438,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""o...","[\n ""重新載入 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ..."
6,240829,3bc6f903023545c0,"[\n ""or.app.start"",\n ""or.app.start"",\n ""or...","[\n ""開啟 App"",\n ""開啟 App"",\n ""點擊外賣自取按鈕"",\n ..."
7,967536,f056f88d-d2ad-4051-af5b-0b81ecd249bf,"[\n ""or.app.resume"",\n ""or.qcksearch"",\n ""o...","[\n ""重新載入 App"",\n ""開啟快速搜尋"",\n ""載入 POI 詳情"",\..."
8,693064,ac224d2f-4326-4a83-b662-ea697ee607b1,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""m...","[\n ""重新載入 App"",\n ""開啟 App"",\n ""從收藏清單搜尋收藏 PO..."
9,900640,dfc1b3ddecfc2e15,"[\n ""or.app.start"",\n ""or.app.start"",\n ""or...","[\n ""開啟 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ""搜..."


In [ ]:
#behavior_tracking.to_csv("behavior_tracking.csv", index=False, encoding="utf-8-sig")

Find unknown event

In [8]:
def list_unknown_events(dataframe):
    normalized_actions = (
        dataframe["EventAction"]
        .fillna("(empty event action)")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    unknown_events = (
        normalized_actions[~normalized_actions.isin(event_mapping)]
        .value_counts()
        .rename_axis("unknown_event_action")
        .reset_index(name="event_count")
    )

    return unknown_events


unknown_events = list_unknown_events(df_bq)
#unknown_events

In [7]:
# locate unknown event
unknown_event_details = tracking_source[
    tracking_source["EventAction"]
    .fillna("(empty event action)")
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(unknown_events["unknown_event_action"])
][
    ["SessionId", "DeviceId", "Time", "EventAction", "EventLabelRaw"]
]

#unknown_event_details

1. 定位or.takeaway.pay 
    開始倒序搜尋

2. 前面滿足四個條件（存在且順序正確，中間可夾雜其他event）：
or.takeaway.order
or.takeaway.checkout
or.takeaway.place-order


or.takeaway.pay

3. 繼續倒序搜尋，最近takeaway.order的entry source

New column: Entry Source

In [9]:
entry_source_mapping = {
    "or.search.quick": "Search",
    "or.qcksearch": "Search",
    "or.search.nearby": "Search",
    "or.search.get-poi": "Search",
    "or.search.layer.record": "Search",
    "or.search.layer": "Search",
    "or.search.layer.tips": "Search",
    
    "or.search.themelist.takeaway": "TakeawayButton",

    "myor.search.bkmpoi": "Bookmark",
    "or.myor.bkmpoi": "Bookmark",

    "or.myor.orderlist": "OrderHistory",

    "or.deeplink.get-poi": "Deeplink",

    "or.app.openpush": "Push",
}

# tracking_source 已按 Time 排序，所以每個 session 的第一個入口事件就是 entry source
tracking_source["normalized_action"] = (
    tracking_source["EventAction"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

def find_entry_source(session_events):
    actions = session_events["normalized_action"].tolist()

    # 由最後一次 payment 開始倒序找，避免 session 內有多次 payment 時取到較早的 journey
    payment_positions = [
        index
        for index, action in enumerate(actions)
        if action == "or.takeaway.pay"
    ]

    for payment_index in reversed(payment_positions):       #place-order
        place_order_index = next(
            (   index
                for index in range(payment_index - 1, -1, -1)
                if actions[index] == "or.takeaway.place-order"),
            None
        )
        if place_order_index is None:
            continue

        checkout_index = next(          # checkout
            (   index
                for index in range(place_order_index - 1, -1, -1)
                if actions[index] == "or.takeaway.checkout"),
            None
        )
        if checkout_index is None:
            continue

        takeaway_order_index = next(           # takeaway order
            (   index
                for index in range(checkout_index - 1, -1, -1)
                if actions[index] == "or.takeaway.order"),
            None
        )
        if takeaway_order_index is None:
            continue

        # 已確認完整下單順序；在該 takeaway order 找最近entry source
        for index in range(takeaway_order_index - 1, -1, -1):
            entry_source = entry_source_mapping.get(actions[index])
            if entry_source is not None:
                return entry_source

        # 有完整下單流程，但 takeaway order 前沒有可識別入口
        return pd.NA

    # 找不到完整的 order -> checkout -> place-order -> pay 流程
    return pd.NA

session_entry_source = (
    tracking_source
    .groupby(["SessionId", "DeviceId"], sort=False)
    .apply(find_entry_source)
    .rename("entry_source")
    .reset_index()
)
behavior_tracking = behavior_tracking.drop(
    columns=["entry_source"],
    errors="ignore"
)
behavior_tracking = behavior_tracking.merge(
    session_entry_source,
    on=["SessionId", "DeviceId"],
    how="left"
)

behavior_tracking

,SessionId,DeviceId,event_timeline,journey,entry_source
0,178112,2c1ec027-be02-4201-a8ce-09c4ca6031b2,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""o...","[\n ""重新載入 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ...",Search
1,127050,1f894cab868b116f,"[\n ""or.poi.get-overview"",\n ""or.poi.get-det...","[\n ""載入 POI 概覽"",\n ""載入 POI 詳情"",\n ""載入 POI 詳...",Search
2,925466,e5f59098851843a7,"[\n ""or.app.start"",\n ""or.app.start"",\n ""or...","[\n ""開啟 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ""搜...",Search
3,119842,1daf763b-8da1-4b2f-b824-12d558a8ffda,"[\n ""or.app.resume"",\n ""impression.poi"",\n ...","[\n ""重新載入 App"",\n ""瀏覽POI"",\n ""瀏覽POI"",\n ""瀏...",NaN
4,252253,3ea40686-e25e-4ccb-9887-64c71db37c78,"[\n ""or.app.resume"",\n ""or.qcksearch"",\n ""o...","[\n ""重新載入 App"",\n ""開啟快速搜尋"",\n ""載入 POI 詳情"",\...",Search
5,818327,cb512dc0-6f1c-4824-8d2e-e08565916438,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""o...","[\n ""重新載入 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ...",Search
6,240829,3bc6f903023545c0,"[\n ""or.app.start"",\n ""or.app.start"",\n ""or...","[\n ""開啟 App"",\n ""開啟 App"",\n ""點擊外賣自取按鈕"",\n ...",TakeawayButton
7,967536,f056f88d-d2ad-4051-af5b-0b81ecd249bf,"[\n ""or.app.resume"",\n ""or.qcksearch"",\n ""o...","[\n ""重新載入 App"",\n ""開啟快速搜尋"",\n ""載入 POI 詳情"",\...",Search
8,693064,ac224d2f-4326-4a83-b662-ea697ee607b1,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""m...","[\n ""重新載入 App"",\n ""開啟 App"",\n ""從收藏清單搜尋收藏 PO...",Bookmark
9,900640,dfc1b3ddecfc2e15,"[\n ""or.app.start"",\n ""or.app.start"",\n ""or...","[\n ""開啟 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ""搜...",Search


In [ ]:
#behavior_tracking.to_csv("entry_source.csv", index=False, encoding="utf-8-sig")